# Time To Entry Experiments

Ce notebook reprend le script `time_to_entry_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Modele le temps restant avant entree physique avec evaluation causale.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Train multi-bin and survival-style time-to-entry early-warning models.
- Run par defaut : `runs/exp_101_time_to_entry_multibin_survival_seq30`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "time_to_entry_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from ml_pipeline import ROOT, RUNS_DIR, alarm_episodes, safe_auc, write_json
from sequence_experiments import make_model, resample_sequence_np, set_seed


TIME_BINS = [
    ("no_entry_or_near_miss", None),
    ("post_entry", (-math.inf, 0.0)),
    ("pre_00_03", (0.0, 0.3)),
    ("pre_03_05", (0.3, 0.5)),
    ("pre_05_07", (0.5, 0.7)),
    ("pre_07_10", (0.7, 1.0)),
    ("pre_10_15", (1.0, 1.5)),
    ("pre_gt_15", (1.5, math.inf)),
]

SURVIVAL_HORIZONS = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `create_run_dir`

Cette cellule definit `create_run_dir`. Elle prepare une partie du script.

In [ ]:
def create_run_dir(run_name):
    base = RUNS_DIR / run_name
    run_dir = base
    suffix = 2
    while run_dir.exists():
        run_dir = RUNS_DIR / f"{run_name}_r{suffix:02d}"
        suffix += 1
    for sub in ["features", "models", "metrics", "error_review"]:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)
    return run_dir


## Fonction `device_from_arg`

Cette cellule definit `device_from_arg`. Elle prepare une partie du script.

In [ ]:
def device_from_arg(arg):
    if arg == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(arg)


## Fonction `load_sequence_data`

Cette cellule definit `load_sequence_data`. Elle prepare une partie du script.

In [ ]:
def load_sequence_data(sequence_run, base_run):
    sequence_run = resolve(sequence_run)
    base_run = resolve(base_run)
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    X = data["X"].astype(np.float32)
    meta = pd.read_csv(sequence_run / "features" / "sequence_index.csv")

    entry_path = base_run / "features" / "entry_times.csv"
    if entry_path.exists():
        entry = pd.read_csv(entry_path)
        keep = [
            "video_id",
            "event_type",
            "target_source",
            "body_part",
            "spatial_relation",
            "physical_entry_time_s",
            "risk_onset_time_s",
        ]
        keep = [col for col in keep if col in entry.columns]
        meta = meta.merge(entry[keep], on="video_id", how="left")
    else:
        meta["event_type"] = ""

    meta["target_time_numeric"] = pd.to_numeric(meta["target_time_s"], errors="coerce")
    meta["time_to_target_numeric"] = pd.to_numeric(meta["time_to_target_s"], errors="coerce")
    meta["has_physical_entry"] = meta["is_danger_clip"].astype(int).eq(1) & meta["target_time_numeric"].notna()
    meta["is_hard_negative"] = meta.get("event_type", pd.Series("", index=meta.index)).fillna("").eq("near_miss")
    return X, meta, sequence_run, base_run


## Fonction `make_multibin_targets`

Cette cellule definit `make_multibin_targets`. Elle prepare une partie du script.

In [ ]:
def make_multibin_targets(meta):
    y = np.zeros(len(meta), dtype=np.int64)
    tte = meta["time_to_target_numeric"].to_numpy(dtype=np.float32)
    is_entry = meta["has_physical_entry"].to_numpy(dtype=bool)

    y[is_entry & (tte <= 0.0)] = 1
    y[is_entry & (tte > 0.0) & (tte <= 0.3)] = 2
    y[is_entry & (tte > 0.3) & (tte <= 0.5)] = 3
    y[is_entry & (tte > 0.5) & (tte <= 0.7)] = 4
    y[is_entry & (tte > 0.7) & (tte <= 1.0)] = 5
    y[is_entry & (tte > 1.0) & (tte <= 1.5)] = 6
    y[is_entry & (tte > 1.5)] = 7

    weights = np.ones(len(meta), dtype=np.float32)
    weights[meta["is_hard_negative"].to_numpy(dtype=bool)] = 1.50
    weights[is_entry & (tte <= 0.0)] = 0.40
    return y, weights


## Fonction `make_survival_targets`

Cette cellule definit `make_survival_targets`. Elle prepare une partie du script.

In [ ]:
def make_survival_targets(meta):
    y = np.zeros((len(meta), len(SURVIVAL_HORIZONS)), dtype=np.float32)
    tte = meta["time_to_target_numeric"].to_numpy(dtype=np.float32)
    is_entry = meta["has_physical_entry"].to_numpy(dtype=bool)
    future = is_entry & (tte > 0.0)
    for h_idx, horizon in enumerate(SURVIVAL_HORIZONS):
        y[:, h_idx] = (future & (tte <= horizon)).astype(np.float32)

    weights = np.ones(len(meta), dtype=np.float32)
    weights[meta["is_hard_negative"].to_numpy(dtype=bool)] = 1.50
    weights[is_entry & (tte <= 0.0)] = 0.35
    return y, weights


## Fonction `future_entry_target`

Cette cellule definit `future_entry_target`. Elle prepare une partie du script.

In [ ]:
def future_entry_target(meta, max_horizon=1.5, min_horizon=0.0):
    tte = meta["time_to_target_numeric"]
    return (
        meta["has_physical_entry"]
        & tte.gt(min_horizon)
        & tte.le(max_horizon)
    ).to_numpy(dtype=np.int64)


## Classe `SequenceObjectiveDataset`

Cette cellule definit `SequenceObjectiveDataset`. Elle prepare une partie du script.

In [ ]:
class SequenceObjectiveDataset(Dataset):
    def __init__(self, X, y, weights, indices, objective, augment=False, seed=42):
        self.X = X
        self.y = y
        self.weights = weights
        self.indices = np.asarray(indices, dtype=np.int64)
        self.objective = objective
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        source_idx = self.indices[idx]
        x = self.X[source_idx].copy()
        if self.augment:
            x = self.apply_augmentations(x)
        if self.objective == "multibin":
            y = torch.tensor(int(self.y[source_idx]), dtype=torch.long)
        else:
            y = torch.from_numpy(self.y[source_idx].astype(np.float32))
        return (
            torch.from_numpy(x.astype(np.float32)),
            y,
            torch.tensor(float(self.weights[source_idx]), dtype=torch.float32),
        )

    def apply_augmentations(self, x):
        if self.rng.random() < 0.85:
            x += self.rng.normal(0.0, 0.035, size=x.shape).astype(np.float32)
        if self.rng.random() < 0.35:
            n_features = max(1, int(x.shape[1] * self.rng.uniform(0.03, 0.10)))
            cols = self.rng.choice(x.shape[1], size=n_features, replace=False)
            x[:, cols] = 0.0
        if self.rng.random() < 0.35:
            width = int(self.rng.integers(2, max(3, x.shape[0] // 4)))
            start = int(self.rng.integers(0, max(1, x.shape[0] - width + 1)))
            x[start : start + width] = 0.0
        if self.rng.random() < 0.35:
            x = resample_sequence_np(x, float(self.rng.uniform(0.85, 1.18)))
        return x


## Fonction `class_weights`

Cette cellule definit `class_weights`. Elle prepare une partie du script.

In [ ]:
def class_weights(y, train_idx, n_classes):
    counts = np.bincount(y[train_idx], minlength=n_classes).astype(np.float32)
    total = counts.sum()
    weights = total / np.maximum(counts, 1.0) / n_classes
    weights = np.clip(weights, 0.25, 12.0)
    return weights.astype(np.float32), counts.astype(int)


## Fonction `survival_pos_weights`

Cette cellule definit `survival_pos_weights`. Elle prepare une partie du script.

In [ ]:
def survival_pos_weights(y, train_idx):
    pos = y[train_idx].sum(axis=0)
    neg = len(train_idx) - pos
    weights = neg / np.maximum(pos, 1.0)
    return np.clip(weights, 1.0, 20.0).astype(np.float32), pos.astype(int)


## Fonction `multibin_loss`

Cette cellule definit `multibin_loss`. Elle prepare une partie du script.

In [ ]:
def multibin_loss(logits, targets, sample_weights, class_weight, label_smoothing):
    losses = F.cross_entropy(
        logits,
        targets,
        weight=class_weight,
        reduction="none",
        label_smoothing=label_smoothing,
    )
    return (losses * sample_weights).sum() / sample_weights.sum().clamp(min=1.0)


## Fonction `survival_loss`

Cette cellule definit `survival_loss`. Elle prepare une partie du script.

In [ ]:
def survival_loss(logits, targets, sample_weights, pos_weight, focal_gamma):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none")
    if focal_gamma > 0:
        pt = torch.exp(-bce).clamp(min=1e-6, max=1.0)
        bce = ((1.0 - pt) ** focal_gamma) * bce
    weighted = bce * sample_weights.view(-1, 1)
    denom = sample_weights.sum().clamp(min=1.0) * targets.shape[1]
    return weighted.sum() / denom


## Fonction `predict_logits`

Cette cellule definit `predict_logits`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict_logits(model, X, batch_size, device):
    model.eval()
    rows = []
    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[start : start + batch_size]).to(device)
        rows.append(model(xb).detach().cpu())
    return torch.cat(rows, dim=0).numpy()


## Fonction `multibin_score_frame`

Cette cellule definit `multibin_score_frame`. Elle prepare une partie du script.

In [ ]:
def multibin_score_frame(meta, probs, model_name, train_time_s, inference_s):
    pred = meta.copy()
    pred["objective"] = "multibin"
    pred["model_name"] = model_name
    pred["train_time_s"] = float(train_time_s)
    pred["inference_ms_per_window"] = float(inference_s * 1000.0 / max(len(pred), 1))
    for idx, (name, _) in enumerate(TIME_BINS):
        pred[f"p_{name}"] = probs[:, idx]
    pred["score_pre_any"] = probs[:, 2:].sum(axis=1)
    pred["score_pre_ge03"] = probs[:, 3:].sum(axis=1)
    pred["score_pre_ge05"] = probs[:, 4:].sum(axis=1)
    pred["score_pre_03_15"] = probs[:, 3:7].sum(axis=1)
    pred["score_pre_05_15"] = probs[:, 4:7].sum(axis=1)
    pred["score_pre_00_10"] = probs[:, 2:6].sum(axis=1)
    return pred


## Fonction `survival_score_frame`

Cette cellule definit `survival_score_frame`. Elle prepare une partie du script.

In [ ]:
def survival_score_frame(meta, probs, model_name, train_time_s, inference_s):
    pred = meta.copy()
    pred["objective"] = "survival"
    pred["model_name"] = model_name
    pred["train_time_s"] = float(train_time_s)
    pred["inference_ms_per_window"] = float(inference_s * 1000.0 / max(len(pred), 1))
    for idx, horizon in enumerate(SURVIVAL_HORIZONS):
        pred[f"p_entry_by_{horizon:.1f}s"] = probs[:, idx]
    h = {horizon: probs[:, idx] for idx, horizon in enumerate(SURVIVAL_HORIZONS)}
    pred["score_by_03"] = h[0.3]
    pred["score_by_05"] = h[0.5]
    pred["score_by_07"] = h[0.7]
    pred["score_by_10"] = h[1.0]
    pred["score_by_15"] = h[1.5]
    pred["score_by_20"] = h[2.0]
    pred["score_future_03_15"] = np.maximum(h[1.5] - h[0.3], 0.0)
    pred["score_future_05_20"] = np.maximum(h[2.0] - h[0.5], 0.0)
    pred["score_mean_07_15"] = (h[0.7] + h[1.0] + h[1.5]) / 3.0
    return pred


## Fonction `validation_score`

Cette cellule definit `validation_score`. Elle prepare une partie du script.

In [ ]:
def validation_score(objective, logits, meta, val_idx):
    val_meta = meta.iloc[val_idx]
    target = future_entry_target(val_meta, max_horizon=1.5, min_horizon=0.0)
    if objective == "multibin":
        probs = torch.softmax(torch.from_numpy(logits[val_idx]), dim=1).numpy()
        score = probs[:, 3:7].sum(axis=1)
    else:
        probs = 1.0 / (1.0 + np.exp(-logits[val_idx]))
        score = probs[:, SURVIVAL_HORIZONS.index(1.0)]
    return float(safe_auc(average_precision_score, target, score) or 0.0)


## Fonction `train_one`

Cette cellule definit `train_one`. Elle prepare une partie du script.

In [ ]:
def train_one(objective, kind, name, X, y, sample_weights, meta, run_dir, args, device):
    seq_len, input_dim = X.shape[1], X.shape[2]
    out_dim = len(TIME_BINS) if objective == "multibin" else len(SURVIVAL_HORIZONS)
    train_idx = np.flatnonzero(meta["split"].to_numpy() == "train")
    val_idx = np.flatnonzero(meta["split"].to_numpy() == "val")

    train_ds = SequenceObjectiveDataset(X, y, sample_weights, train_idx, objective, augment=True, seed=args.seed)
    val_ds = SequenceObjectiveDataset(X, y, sample_weights, val_idx, objective, augment=False, seed=args.seed)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)

    model = make_model(kind, seq_len, input_dim, out_dim).to(device)
    if objective == "multibin":
        cw_np, target_counts = class_weights(y, train_idx, len(TIME_BINS))
        class_weight = torch.tensor(cw_np, dtype=torch.float32, device=device)
        pos_weight = None
    else:
        pw_np, target_counts = survival_pos_weights(y, train_idx)
        pos_weight = torch.tensor(pw_np, dtype=torch.float32, device=device)
        class_weight = None

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best = {"score": -1.0, "epoch": 0, "state": None}
    history = []
    patience_left = args.patience
    start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb, wb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            wb = wb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            if objective == "multibin":
                loss = multibin_loss(logits, yb, wb, class_weight, args.label_smoothing)
            else:
                loss = survival_loss(logits, yb, wb, pos_weight, args.focal_gamma)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        val_logits = []
        model.eval()
        with torch.no_grad():
            for xb, _, _ in val_loader:
                val_logits.append(model(xb.to(device)).detach().cpu().numpy())
        val_logits = np.vstack(val_logits)
        full_val_logits = np.zeros((len(meta), out_dim), dtype=np.float32)
        full_val_logits[val_idx] = val_logits
        val_score = validation_score(objective, full_val_logits, meta, val_idx)
        scheduler.step(val_score)

        row = {
            "objective": objective,
            "model": name,
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "val_ap_future_0_15s": val_score,
            "lr": float(optimizer.param_groups[0]["lr"]),
        }
        history.append(row)
        if val_score > best["score"] + 1e-5:
            best = {
                "score": val_score,
                "epoch": epoch,
                "state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            }
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time_s = time.perf_counter() - start
    model_path = run_dir / "models" / f"{name}.pt"
    torch.save(
        {
            "objective": objective,
            "model_name": name,
            "kind": kind,
            "state_dict": model.state_dict(),
            "seq_len": seq_len,
            "input_dim": input_dim,
            "out_dim": out_dim,
            "time_bins": [name for name, _ in TIME_BINS],
            "survival_horizons": SURVIVAL_HORIZONS,
            "best_epoch": best["epoch"],
            "best_val_ap_future_0_15s": best["score"],
            "target_counts": [int(x) for x in target_counts],
        },
        model_path,
    )
    return model, history, train_time_s, int(model_path.stat().st_size), best


## Fonction `evaluate_causal`

Cette cellule definit `evaluate_causal`. Elle prepare une partie du script.

In [ ]:
def evaluate_causal(pred, score_col, threshold, split, persistence_windows):
    sdf = pred[pred["split"].eq(split)].copy()
    pre = early03 = early05 = fp = danger = missed = 0
    neg_min = 0.0
    early_times = []
    for _, group in sdf.groupby("video_id", sort=False):
        group = group.sort_values("time_s")
        alarms = alarm_episodes(
            group["time_s"],
            group[score_col],
            threshold,
            gap_s=1.0,
            persistence_windows=persistence_windows,
        )
        is_danger = int(group["is_danger_clip"].max()) == 1
        target_values = pd.to_numeric(group["target_time_s"], errors="coerce").dropna()
        target = float(target_values.iloc[0]) if len(target_values) else math.nan
        if is_danger and not math.isnan(target):
            danger += 1
            pre_alarms = [float(t) for t in alarms if float(t) < target]
            if pre_alarms:
                first = min(pre_alarms)
                lead = target - first
                early_times.append(lead)
                pre += 1
                if lead >= 0.3:
                    early03 += 1
                if lead >= 0.5:
                    early05 += 1
            else:
                missed += 1
        else:
            fp += len(alarms)
            if len(group):
                neg_min += max(0.0, float(group["time_s"].max() - group["time_s"].min())) / 60.0

    precision = pre / (pre + fp) if (pre + fp) else np.nan
    pre_recall = pre / danger if danger else np.nan
    early03_recall = early03 / danger if danger else np.nan
    early05_recall = early05 / danger if danger else np.nan
    f1 = 2 * precision * pre_recall / (precision + pre_recall) if precision == precision and (precision + pre_recall) > 0 else np.nan
    return {
        "score_col": score_col,
        "threshold": float(threshold),
        "split": split,
        "persistence_windows": int(persistence_windows),
        "danger_videos": int(danger),
        "pre_entry_detected": int(pre),
        "early_03_detected": int(early03),
        "early_05_detected": int(early05),
        "missed_entries": int(missed),
        "false_alarm_episodes": int(fp),
        "negative_minutes": float(neg_min),
        "pre_entry_recall": float(pre_recall),
        "early_03_recall": float(early03_recall),
        "early_05_recall": float(early05_recall),
        "event_precision": float(precision),
        "event_f1": float(f1),
        "false_alarms_per_min": float(fp / neg_min) if neg_min > 0 else 0.0,
        "median_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
        "mean_early_warning_s": float(np.mean(early_times)) if early_times else np.nan,
    }


## Fonction `add_selection_score`

Cette cellule definit `add_selection_score`. Elle prepare une partie du script.

In [ ]:
def add_selection_score(df):
    out = df.copy()
    for col in ["pre_entry_recall", "early_03_recall", "early_05_recall", "event_precision", "false_alarms_per_min"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["selection_score"] = (
        1.8 * out["early_03_recall"].fillna(0.0)
        + 1.2 * out["early_05_recall"].fillna(0.0)
        + 0.5 * out["pre_entry_recall"].fillna(0.0)
        + 0.8 * out["event_precision"].fillna(0.0)
        - 0.06 * out["false_alarms_per_min"].fillna(0.0).clip(upper=20.0)
    )
    return out


## Fonction `evaluate_prediction_frame`

Cette cellule definit `evaluate_prediction_frame`. Elle prepare une partie du script.

In [ ]:
def evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values):
    rows = []
    for score_col in score_cols:
        for threshold in thresholds:
            for persistence in persistence_values:
                for split in ["val", "test"]:
                    row = evaluate_causal(pred, score_col, threshold, split, persistence)
                    row["objective"] = str(pred["objective"].iloc[0])
                    row["model_name"] = str(pred["model_name"].iloc[0])
                    row["inference_ms_per_window"] = float(pred["inference_ms_per_window"].iloc[0])
                    row["train_time_s"] = float(pred["train_time_s"].iloc[0])
                    rows.append(row)
    return rows


## Fonction `score_columns_for_objective`

Cette cellule definit `score_columns_for_objective`. Elle prepare une partie du script.

In [ ]:
def score_columns_for_objective(objective):
    if objective == "multibin":
        return [
            "score_pre_any",
            "score_pre_ge03",
            "score_pre_ge05",
            "score_pre_03_15",
            "score_pre_05_15",
            "score_pre_00_10",
        ]
    if objective == "survival":
        return [
            "score_by_03",
            "score_by_05",
            "score_by_07",
            "score_by_10",
            "score_by_15",
            "score_by_20",
            "score_future_03_15",
            "score_future_05_20",
            "score_mean_07_15",
        ]
    return ["prob_0.5s", "prob_1.0s", "prob_1.5s", "prob_2.0s"]


## Fonction `evaluate_baseline_sequence_predictions`

Cette cellule definit `evaluate_baseline_sequence_predictions`. Elle prepare une partie du script.

In [ ]:
def evaluate_baseline_sequence_predictions(sequence_run, thresholds, persistence_values):
    rows = []
    feature_dir = sequence_run / "features"
    for path in sorted(feature_dir.glob("predictions_*.csv")):
        pred = pd.read_csv(path)
        pred["objective"] = "baseline_horizon"
        pred["model_name"] = path.stem.replace("predictions_", "baseline_")
        pred["train_time_s"] = 0.0
        pred["inference_ms_per_window"] = 0.0
        score_cols = [col for col in score_columns_for_objective("baseline_horizon") if col in pred.columns]
        rows.extend(evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values))
    return rows


## Fonction `ap_auc_rows`

Cette cellule definit `ap_auc_rows`. Elle prepare une partie du script.

In [ ]:
def ap_auc_rows(pred, score_cols):
    rows = []
    for split in ["val", "test"]:
        sdf = pred[pred["split"].eq(split)]
        y03 = future_entry_target(sdf, max_horizon=1.5, min_horizon=0.3)
        y05 = future_entry_target(sdf, max_horizon=1.5, min_horizon=0.5)
        for score_col in score_cols:
            rows.append(
                {
                    "objective": str(pred["objective"].iloc[0]),
                    "model_name": str(pred["model_name"].iloc[0]),
                    "score_col": score_col,
                    "split": split,
                    "ap_pre_03_15": float(safe_auc(average_precision_score, y03, sdf[score_col]) or 0.0),
                    "roc_pre_03_15": float(safe_auc(roc_auc_score, y03, sdf[score_col]) or 0.0),
                    "ap_pre_05_15": float(safe_auc(average_precision_score, y05, sdf[score_col]) or 0.0),
                    "roc_pre_05_15": float(safe_auc(roc_auc_score, y05, sdf[score_col]) or 0.0),
                }
            )
    return rows


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, metrics, ap_metrics, config, selected_val, selected_test, best_test, baseline_best_test):
    lines = [
        "# Time-to-Entry Early-Warning Experiments",
        "",
        f"_Updated: {datetime.now().isoformat(timespec='seconds')}_",
        "",
        "This run trains two target families on the existing 30-frame causal pose windows:",
        "",
        "- `multibin`: one class for no-entry/near-miss, one post-entry class, and ordered pre-entry bins.",
        "- `survival`: independent horizon heads for whether physical entry will occur within 0.3, 0.5, 0.7, 1.0, 1.5, or 2.0 seconds.",
        "",
        "The evaluator is live-style: at timestamp `t`, the score only sees frames up to `t`; a true alarm must start before the physical entry timestamp.",
        "",
        "## Configuration",
        "",
        f"- Sequence run: `{config['sequence_run']}`",
        f"- Base annotation run: `{config['base_run']}`",
        f"- Device: `{config['device']}`",
        f"- Epoch cap / patience: `{config['epochs']}` / `{config['patience']}`",
        f"- Thresholds: `{config['thresholds']}`",
        f"- Persistence windows: `{config['persistence_values']}`",
        "",
        "## Validation-Selected Operating Point",
        "",
        "| field | value |",
        "|---|---:|",
    ]
    for key in [
        "objective",
        "model_name",
        "score_col",
        "threshold",
        "persistence_windows",
        "selection_score",
        "pre_entry_recall",
        "early_03_recall",
        "early_05_recall",
        "event_precision",
        "false_alarms_per_min",
        "median_early_warning_s",
    ]:
        value = selected_val.get(key, "")
        if isinstance(value, float):
            value = f"{value:.3f}"
        lines.append(f"| val {key} | {value} |")
    lines.extend(["", "## Same Policy On Test", "", "| metric | value |", "|---|---:|"])
    for key in [
        "danger_videos",
        "pre_entry_detected",
        "early_03_detected",
        "early_05_detected",
        "false_alarm_episodes",
        "pre_entry_recall",
        "early_03_recall",
        "early_05_recall",
        "event_precision",
        "false_alarms_per_min",
        "median_early_warning_s",
        "inference_ms_per_window",
    ]:
        value = selected_test.get(key, "")
        if isinstance(value, float):
            value = f"{value:.3f}"
        lines.append(f"| {key} | {value} |")

    lines.extend(["", "## Best Test Rows Found", ""])
    lines.append("| objective | model | score | threshold | persist | pre recall | >=0.3s | >=0.5s | precision | FA/min | median early |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for _, row in best_test.head(12).iterrows():
        lines.append(
            f"| {row['objective']} | {row['model_name']} | {row['score_col']} | {row['threshold']:.2f} | "
            f"{int(row['persistence_windows'])} | {row['pre_entry_recall']:.3f} | {row['early_03_recall']:.3f} | "
            f"{row['early_05_recall']:.3f} | {row['event_precision']:.3f} | {row['false_alarms_per_min']:.3f} | "
            f"{row['median_early_warning_s']:.3f} |"
        )

    if baseline_best_test is not None and len(baseline_best_test):
        lines.extend(["", "## Same-Split Horizon Baseline Rows", ""])
        lines.append("| model | score | threshold | persist | pre recall | >=0.3s | >=0.5s | precision | FA/min |")
        lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|")
        for _, row in baseline_best_test.head(8).iterrows():
            lines.append(
                f"| {row['model_name']} | {row['score_col']} | {row['threshold']:.2f} | {int(row['persistence_windows'])} | "
                f"{row['pre_entry_recall']:.3f} | {row['early_03_recall']:.3f} | {row['early_05_recall']:.3f} | "
                f"{row['event_precision']:.3f} | {row['false_alarms_per_min']:.3f} |"
            )

    lines.extend(["", "## Artifacts", ""])
    lines.append(f"- Causal metrics: `{run_dir / 'metrics' / 'time_to_entry_causal_metrics.csv'}`")
    lines.append(f"- Ranking/AP metrics: `{run_dir / 'metrics' / 'time_to_entry_ap_metrics.csv'}`")
    lines.append(f"- Training history: `{run_dir / 'metrics' / 'time_to_entry_training_history.csv'}`")
    lines.append(f"- Predictions: `{run_dir / 'features'}`")

    (run_dir / "time_to_entry_experiment_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    set_seed(args.seed)
    device = device_from_arg(args.device)
    X, meta, sequence_run, base_run = load_sequence_data(args.sequence_run, args.base_run)
    run_dir = create_run_dir(args.run_name)
    thresholds = [round(float(x), 2) for x in np.arange(args.threshold_min, args.threshold_max + 1e-9, args.threshold_step)]
    persistence_values = [int(x) for x in args.persistence_windows]

    write_json(
        run_dir / "config.json",
        {
            "sequence_run": str(sequence_run),
            "base_run": str(base_run),
            "device": str(device),
            "seed": args.seed,
            "epochs": args.epochs,
            "patience": args.patience,
            "batch_size": args.batch_size,
            "lr": args.lr,
            "weight_decay": args.weight_decay,
            "label_smoothing": args.label_smoothing,
            "focal_gamma": args.focal_gamma,
            "thresholds": thresholds,
            "persistence_values": persistence_values,
            "time_bins": [name for name, _ in TIME_BINS],
            "survival_horizons": SURVIVAL_HORIZONS,
        },
    )

    multibin_y, multibin_weights = make_multibin_targets(meta)
    survival_y, survival_weights = make_survival_targets(meta)
    target_audit = {
        "rows": int(len(meta)),
        "sequence_shape": [int(x) for x in X.shape],
        "video_counts_by_split": meta.groupby("split")["video_id"].nunique().to_dict(),
        "window_counts_by_split": meta["split"].value_counts().to_dict(),
        "multibin_counts": dict(zip([name for name, _ in TIME_BINS], [int(x) for x in np.bincount(multibin_y, minlength=len(TIME_BINS))])),
        "survival_positive_counts": {f"{h:.1f}s": int(survival_y[:, idx].sum()) for idx, h in enumerate(SURVIVAL_HORIZONS)},
        "hard_negative_windows": int(meta["is_hard_negative"].sum()),
    }
    write_json(run_dir / "metrics" / "time_to_entry_target_audit.json", target_audit)

    specs = []
    for kind in args.model_kinds:
        specs.append(("multibin", kind, f"multibin_{kind}_aug"))
    for kind in args.model_kinds:
        specs.append(("survival", kind, f"survival_{kind}_aug"))

    all_metric_rows = []
    all_ap_rows = []
    all_history = []
    train_rows = []
    for objective, kind, name in specs:
        y = multibin_y if objective == "multibin" else survival_y
        sample_weights = multibin_weights if objective == "multibin" else survival_weights
        print(f"training {name} on {device}")
        model, history, train_time_s, model_size_bytes, best = train_one(
            objective, kind, name, X, y, sample_weights, meta, run_dir, args, device
        )
        all_history.extend(history)
        start = time.perf_counter()
        logits = predict_logits(model, X, args.batch_size, device)
        inference_s = time.perf_counter() - start
        if objective == "multibin":
            probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
            pred = multibin_score_frame(meta, probs, name, train_time_s, inference_s)
        else:
            probs = 1.0 / (1.0 + np.exp(-logits))
            pred = survival_score_frame(meta, probs, name, train_time_s, inference_s)

        pred.to_csv(run_dir / "features" / f"predictions_{name}.csv", index=False)
        score_cols = score_columns_for_objective(objective)
        all_metric_rows.extend(evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values))
        all_ap_rows.extend(ap_auc_rows(pred, score_cols))
        train_rows.append(
            {
                "objective": objective,
                "model_name": name,
                "kind": kind,
                "best_epoch": int(best["epoch"]),
                "best_val_ap_future_0_15s": float(best["score"]),
                "train_time_s": float(train_time_s),
                "model_size_bytes": int(model_size_bytes),
                "inference_ms_per_window": float(inference_s * 1000.0 / max(len(meta), 1)),
            }
        )

    baseline_rows = []
    if args.include_sequence_baseline:
        baseline_rows = evaluate_baseline_sequence_predictions(sequence_run, thresholds, persistence_values)
        all_metric_rows.extend(baseline_rows)

    metrics = add_selection_score(pd.DataFrame(all_metric_rows))
    ap_metrics = pd.DataFrame(all_ap_rows)
    history = pd.DataFrame(all_history)
    training = pd.DataFrame(train_rows)
    metrics.to_csv(run_dir / "metrics" / "time_to_entry_causal_metrics.csv", index=False)
    ap_metrics.to_csv(run_dir / "metrics" / "time_to_entry_ap_metrics.csv", index=False)
    history.to_csv(run_dir / "metrics" / "time_to_entry_training_history.csv", index=False)
    training.to_csv(run_dir / "metrics" / "time_to_entry_training_summary.csv", index=False)

    new_metrics = metrics[~metrics["objective"].eq("baseline_horizon")].copy()
    val_rank = new_metrics[new_metrics["split"].eq("val")].sort_values("selection_score", ascending=False)
    selected_val = val_rank.iloc[0].to_dict()
    selected_test_df = new_metrics[
        new_metrics["split"].eq("test")
        & new_metrics["objective"].eq(selected_val["objective"])
        & new_metrics["model_name"].eq(selected_val["model_name"])
        & new_metrics["score_col"].eq(selected_val["score_col"])
        & new_metrics["threshold"].eq(selected_val["threshold"])
        & new_metrics["persistence_windows"].eq(selected_val["persistence_windows"])
    ]
    selected_test = selected_test_df.iloc[0].to_dict() if len(selected_test_df) else {}
    best_test = new_metrics[new_metrics["split"].eq("test")].sort_values("selection_score", ascending=False)
    baseline_best_test = None
    if len(baseline_rows):
        baseline_best_test = metrics[(metrics["objective"].eq("baseline_horizon")) & (metrics["split"].eq("test"))].sort_values("selection_score", ascending=False)

    write_json(run_dir / "metrics" / "time_to_entry_selected_policy.json", {"val": selected_val, "test": selected_test})
    write_summary(
        run_dir,
        metrics,
        ap_metrics,
        {
            "sequence_run": str(sequence_run),
            "base_run": str(base_run),
            "device": str(device),
            "epochs": args.epochs,
            "patience": args.patience,
            "thresholds": thresholds,
            "persistence_values": persistence_values,
        },
        selected_val,
        selected_test,
        best_test,
        baseline_best_test,
    )
    print(run_dir)
    print(run_dir / "time_to_entry_experiment_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Train multi-bin and survival-style time-to-entry early-warning models.")
    parser.add_argument("--sequence-run", default="runs/exp_072_physical_entry_seq30_focal")
    parser.add_argument("--base-run", default="runs/exp_070_physical_entry_baseline")
    parser.add_argument("--run-name", default="exp_101_time_to_entry_multibin_survival_seq30")
    parser.add_argument("--model-kinds", nargs="+", default=["tcn", "cnn1d", "gru", "cnn_gru"])
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--patience", type=int, default=6)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--focal-gamma", type=float, default=1.5)
    parser.add_argument("--threshold-min", type=float, default=0.05)
    parser.add_argument("--threshold-max", type=float, default=0.95)
    parser.add_argument("--threshold-step", type=float, default=0.05)
    parser.add_argument("--persistence-windows", nargs="+", type=int, default=[1, 2])
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--include-sequence-baseline", action="store_true", default=True)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_101_time_to_entry_multibin_survival_seq30_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["time_to_entry_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
